In [2]:
import sys
print(sys.executable)


c:\Users\Ferryadmin\Projects\data_science_projects\Electricity_Renewables_Project\env\Scripts\python.exe


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Notebook is working")

Notebook is working


In [4]:
import os

os.chdir("..")
print(os.getcwd())

c:\Users\Ferryadmin\Projects\data_science_projects\Electricity_Renewables_Project


In [5]:
with open("data/interim/electricity_prices.csv", "r", encoding="utf-8", errors="replace") as f:
    for i in range(15):
        print(f.readline())

"Average retail price of electricity"

"https://www.eia.gov/electricity/data/browser/#/topic/7?agg=0,1&geo=fvvvvvvvvvvvo&endsec=6&freq=M&start=200101&end=202601&ctype=linechart&ltype=pin&rtype=s&maptype=0&rse=0&pin="

"Tue Apr 14 2026 18:05:54 GMT-0400 (Bolivia Time)"

"Source: U.S. Energy Information Administration"

"description","units","source key","Jan 2001","Feb 2001","Mar 2001","Apr 2001","May 2001","Jun 2001","Jul 2001","Aug 2001","Sep 2001","Oct 2001","Nov 2001","Dec 2001","Jan 2002","Feb 2002","Mar 2002","Apr 2002","May 2002","Jun 2002","Jul 2002","Aug 2002","Sep 2002","Oct 2002","Nov 2002","Dec 2002","Jan 2003","Feb 2003","Mar 2003","Apr 2003","May 2003","Jun 2003","Jul 2003","Aug 2003","Sep 2003","Oct 2003","Nov 2003","Dec 2003","Jan 2004","Feb 2004","Mar 2004","Apr 2004","May 2004","Jun 2004","Jul 2004","Aug 2004","Sep 2004","Oct 2004","Nov 2004","Dec 2004","Jan 2005","Feb 2005","Mar 2005","Apr 2005","May 2005","Jun 2005","Jul 2005","Aug 2005","Sep 2005","Oct 2005","Nov 20

In [6]:
import pandas as pd

df = pd.read_csv("data/interim/electricity_prices.csv", skiprows=4)

print(df.head(10))
print(df.columns)
print(df.shape)

                           description                   units  \
0  Average retail price of electricity  cents per kilowatthour   
1                        United States                     NaN   
2                          New England                     NaN   
3            New England : all sectors                     NaN   
4             New England : commercial  cents per kilowatthour   
5             New England : industrial  cents per kilowatthour   
6                          Connecticut                     NaN   
7            Connecticut : all sectors                     NaN   
8             Connecticut : commercial  cents per kilowatthour   
9             Connecticut : industrial  cents per kilowatthour   

             source key  Jan 2001  Feb 2001  Mar 2001  Apr 2001  May 2001  \
0                   NaN       NaN       NaN       NaN       NaN       NaN   
1   ELEC.PRICE.US-ALL.M       NaN       NaN       NaN       NaN       NaN   
2  ELEC.PRICE.NEW-ALL.M       NaN       Na

In [7]:
df_clean = df[df["description"].str.contains(": commercial|: industrial", na=False)].copy()

df_clean[["description"]].head(10)

,description
4,New England : commercial
5,New England : industrial
8,Connecticut : commercial
9,Connecticut : industrial
12,Maine : commercial
13,Maine : industrial
16,Massachusetts : commercial
17,Massachusetts : industrial
20,New Hampshire : commercial
21,New Hampshire : industrial


In [8]:
df_clean[["state_or_region", "sector"]] = df_clean["description"].str.split(" : ", expand=True)

df_clean[["state_or_region", "sector"]].head()

,state_or_region,sector
4,New England,commercial
5,New England,industrial
8,Connecticut,commercial
9,Connecticut,industrial
12,Maine,commercial


In [9]:
month_columns = df_clean.columns[3:]

df_long = df_clean.melt(
    id_vars=["state_or_region", "sector"],
    value_vars=month_columns,
    var_name="date",
    value_name="price"
)

df_long.head()

,state_or_region,sector,date,price
0,New England,commercial,Jan 2001,10.30
1,New England,industrial,Jan 2001,9.04
2,Connecticut,commercial,Jan 2001,9.19
3,Connecticut,industrial,Jan 2001,8.09
4,Maine,commercial,Jan 2001,10.33


In [ ]:
#Converting monthly date format to yearly
month_columns = df_clean.columns[3:]

df_long = df_clean.melt(
    id_vars=["state_or_region", "sector"],
    value_vars=month_columns,
    var_name="date",
    value_name="price"
)

df_long["date"] = pd.to_datetime(df_long["date"], format="%b %Y")
df_long["price"] = pd.to_numeric(df_long["price"], errors="coerce")

In [28]:
#aggregates monthly values to yearly by mean
df_long["year"] = df_long["date"].dt.year

prices_annual = (
    df_long
    .groupby(["state_or_region", "sector", "year"], as_index=False)["price"]
    .mean()
)

prices_annual.head()

,state_or_region,sector,year,price
0,Alabama,commercial,2001,6.549167
1,Alabama,commercial,2002,6.635833
2,Alabama,commercial,2003,6.855000
3,Alabama,commercial,2004,7.118333
4,Alabama,commercial,2005,7.475833


In [29]:
df_long = df_long.dropna(subset=["price"])

In [33]:
print(prices_annual.head(10))
print(prices_annual.shape)
print(prices_annual.dtypes)

  state_or_region      sector  year      price
0         Alabama  commercial  2001   6.549167
1         Alabama  commercial  2002   6.635833
2         Alabama  commercial  2003   6.855000
3         Alabama  commercial  2004   7.118333
4         Alabama  commercial  2005   7.475833
5         Alabama  commercial  2006   8.133333
6         Alabama  commercial  2007   8.685000
7         Alabama  commercial  2008   9.828333
8         Alabama  commercial  2009  10.047500
9         Alabama  commercial  2010  10.176667
(3172, 4)
state_or_region        str
sector                 str
year                 int32
price              float64
dtype: object


  state_or_region      sector  year     price
0         Alabama  commercial  2001  6.549167
1         Alabama  commercial  2002  6.635833
2         Alabama  commercial  2003  6.855000
3         Alabama  commercial  2004  7.118333
4         Alabama  commercial  2005  7.475833
(3172, 4)


In [36]:
regions = [
    "United States",
    "New England",
    "Middle Atlantic",
    "East North Central",
    "West North Central",
    "South Atlantic",
    "East South Central",
    "West South Central",
    "Mountain",
    "Pacific",
    "Contiguous 48 States",
    "U.S. Total"
]

df_state = prices_annual[~prices_annual["state_or_region"].isin(regions)].copy()
df_region = prices_annual[prices_annual["state_or_region"].isin(regions)].copy()

In [37]:
print(df_state["state_or_region"].unique()[:20])
print(df_region["state_or_region"].unique())

<StringArray>
[             'Alabama',               'Alaska',              'Arizona',
             'Arkansas',           'California',             'Colorado',
          'Connecticut',             'Delaware', 'District Of Columbia',
              'Florida',              'Georgia',               'Hawaii',
                'Idaho',             'Illinois',              'Indiana',
                 'Iowa',               'Kansas',             'Kentucky',
            'Louisiana',                'Maine']
Length: 20, dtype: str
<StringArray>
['East North Central', 'East South Central',    'Middle Atlantic',
           'Mountain',        'New England',     'South Atlantic',
 'West North Central', 'West South Central']
Length: 8, dtype: str


In [38]:
df_state.to_csv("data/processed/electricity_prices_states.csv", index=False)
df_region.to_csv("data/processed/electricity_prices_regions.csv", index=False)